# 2. Weather Feature Extraction

This notebook extracts weather features from **ERA5-Land** via Google Earth Engine:
- Temperature (2m)
- Dewpoint temperature
- Wind speed (u/v components)
- Precipitation
- Derived: Relative Humidity, Vapor Pressure Deficit (VPD)

**Data Source:** [ECMWF ERA5-Land Daily](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR)

> **Prerequisite:** Run Notebook 01 first to generate grid cell definitions.

## Google Earth Engine Setup

In [ ]:
!pip install earthengine-api

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project="wildfire-ml-project")

In [ ]:
import ee

# Define California bounding box
california = ee.Geometry.Rectangle([-125, 32, -114, 42])

## ERA5-Land Daily Weather

In [ ]:
era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate("2019-01-01", "2023-12-31")
    .filterBounds(california)
)

print("Number of ERA5 daily images:", era5.size().getInfo())

In [ ]:
lat_min = 32
lon_min = -125
grid_size = 0.25

unique_cells = unique_cells.copy()

unique_cells["center_lat"] = lat_min + (unique_cells["lat_bin"] + 0.5) * grid_size
unique_cells["center_lon"] = lon_min + (unique_cells["lon_bin"] + 0.5) * grid_size

unique_cells.head()

In [ ]:
import ee

# Convert pandas rows to EE Features
def row_to_feature(row):
    point = ee.Geometry.Point([row["center_lon"], row["center_lat"]])
    return ee.Feature(point, {
        "lat_bin": int(row["lat_bin"]),
        "lon_bin": int(row["lon_bin"])
    })

features = [row_to_feature(row) for _, row in unique_cells.iterrows()]

grid_points = ee.FeatureCollection(features)

print("Number of grid points:", grid_points.size().getInfo())

## Sample Day Extraction (Validation)

In [ ]:
sample_day = ee.Date("2019-07-01")

era5_day = (
    era5
    .filterDate(sample_day, sample_day.advance(1, "day"))
    .first()
)

print("ERA5 bands:", era5_day.bandNames().getInfo())

In [ ]:
sampled = era5_day.sampleRegions(
    collection=grid_points,
    scale=10000,  # 10km scale is fine
    geometries=False
)

print("Sampled size:", sampled.size().getInfo())

In [ ]:
sampled_df = pd.DataFrame(
    sampled.getInfo()["features"]
)

print(sampled_df.head())
print("Rows:", len(sampled_df))

In [ ]:
# Extract properties dictionary into dataframe columns
weather_df = pd.json_normalize(sampled_df["properties"])

print(weather_df.head())
print("Columns:", weather_df.columns)
print("Rows:", len(weather_df))

In [ ]:
needed_cols = [
    "temperature_2m",
    "dewpoint_temperature_2m",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "total_precipitation_sum"
]

weather_df = weather_df[needed_cols].copy()

print(weather_df.head())
print("Columns:", weather_df.columns)
print("Rows:", len(weather_df))

## Unit Conversions

In [ ]:
# Convert Kelvin → Celsius
weather_df["temp_c"] = weather_df["temperature_2m"] - 273.15
weather_df["dewpoint_c"] = weather_df["dewpoint_temperature_2m"] - 273.15

# Convert precipitation meters → mm
weather_df["precip_mm"] = weather_df["total_precipitation_sum"] * 1000

In [ ]:
weather_df["wind_speed"] = np.sqrt(
    weather_df["u_component_of_wind_10m"]**2 +
    weather_df["v_component_of_wind_10m"]**2
)

## Derived Variables: Humidity & VPD

In [ ]:
# Saturation vapor pressure
def sat_vapor_pressure(T):
    return 6.112 * np.exp((17.67 * T) / (T + 243.5))

e_T = sat_vapor_pressure(weather_df["temp_c"])
e_Td = sat_vapor_pressure(weather_df["dewpoint_c"])

weather_df["relative_humidity"] = 100 * (e_Td / e_T)

In [ ]:
weather_df["vpd"] = e_T - e_Td

In [ ]:
weather_df[["temp_c", "precip_mm", "wind_speed", "relative_humidity", "vpd"]].head()

In [ ]:
print(grid_points.size().getInfo())

## Batch Export (Year-by-Year)

In [ ]:
# # 1️⃣ Select only required ERA5 bands
# era5_selected = era5.select([
#     "temperature_2m",
#     "dewpoint_temperature_2m",
#     "u_component_of_wind_10m",
#     "v_component_of_wind_10m",
#     "total_precipitation_sum"
# ])

# # 2️⃣ Function to sample one day and attach date
# def sample_image(image):
#     date_str = image.date().format("YYYY-MM-dd")

#     sampled = image.sampleRegions(
#         collection=grid_points,
#         scale=25000,          # Match ERA5 resolution (~25km)
#         geometries=False
#     )

#     return sampled.map(lambda f: f.set("date", date_str))

# # 3️⃣ Export year-by-year (SAFE approach)
# for year in range(2019, 2024):

#     yearly_collection = era5_selected.filterDate(
#         f"{year}-01-01",
#         f"{year}-12-31"
#     )

#     weather_fc = yearly_collection.map(sample_image).flatten()

#     task = ee.batch.Export.table.toDrive(
#         collection=weather_fc,
#         description=f"era5_weather_{year}",
#         folder="Wildfire_Project",
#         fileNamePrefix=f"era5_weather_{year}",
#         fileFormat="CSV"
#     )

#     task.start()

#     print(f"Export task started for {year}")


## Load Exported Weather Data

In [ ]:
# Path to main project folder (where ERA5 exports are stored)
weather_folder = "/content/drive/MyDrive/Wildfire_Project"

# Get only ERA5 files
weather_files = glob.glob(os.path.join(weather_folder, "era5_weather_*.csv"))

print("Weather files found:", weather_files)

# Read and combine
weather_df_list = [pd.read_csv(f) for f in weather_files]
weather_df = pd.concat(weather_df_list, ignore_index=True)

print("Total weather rows:", len(weather_df))

In [ ]:
print(weather_df.columns)
print(weather_df.head())

In [ ]:
weather_df = weather_df.drop(columns=["system:index", ".geo"])
weather_df["date"] = pd.to_datetime(weather_df["date"])

## Process All Weather Data

In [ ]:
import numpy as np

# Convert units
weather_df["temp_c"] = weather_df["temperature_2m"] - 273.15
weather_df["dewpoint_c"] = weather_df["dewpoint_temperature_2m"] - 273.15
weather_df["precip_mm"] = weather_df["total_precipitation_sum"] * 1000

# Wind speed
weather_df["wind_speed"] = np.sqrt(
    weather_df["u_component_of_wind_10m"]**2 +
    weather_df["v_component_of_wind_10m"]**2
)

# Saturation vapor pressure function
def sat_vapor_pressure(T):
    return 6.112 * np.exp((17.67 * T) / (T + 243.5))

e_T = sat_vapor_pressure(weather_df["temp_c"])
e_Td = sat_vapor_pressure(weather_df["dewpoint_c"])

weather_df["relative_humidity"] = 100 * (e_Td / e_T)
weather_df["vpd"] = e_T - e_Td

In [ ]:
weather_df = weather_df[
    ["lat_bin", "lon_bin", "date",
     "temp_c", "precip_mm", "wind_speed",
     "relative_humidity", "vpd"]
].copy()

print(weather_df.head())
print("Rows:", len(weather_df))